[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dxnn1u/slides/blob/main/decks/pytorch-basic/lecture6_pytorch_basic.ipynb)

# [ Lecture 6 ] PyTorch basic — Companion Notebook

This notebook follows the lecture notes section by section. Run the cells from top to bottom.

**Runtime:** works on Google Colab or Kaggle.
- Colab: `Runtime → Change runtime type → T4 GPU` (optional; everything also runs on CPU).
- Kaggle: turn on **Internet** in the notebook settings, otherwise MNIST cannot be downloaded.

## 0. Setup

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device :", device)
if device == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

torch.manual_seed(0)

# Before we begin

## NumPy

### ndarray = data buffer + metadata
Every ndarray has a raw data buffer plus metadata describing how to read it:
- `dtype`: what type each element is
- `shape`: how many elements along each dimension
- `strides`: how far to jump to reach the next element
- `offset`: where the data starts

We will see strides and offset in action once we meet tensors.

In [ ]:
a = np.arange(12, dtype=np.int64).reshape(3, 4)
print(a)
print("dtype  :", a.dtype)
print("shape  :", a.shape)

### Why we like ndarray: speed (vectorization)
The same element-wise addition, written as a Python loop vs. one NumPy call.
NumPy runs the loop in compiled C code, which can also use SIMD instructions.

In [ ]:
n = 1_000_000
x = np.random.rand(n).astype(np.float32)
y = np.random.rand(n).astype(np.float32)

t0 = time.perf_counter()
z_loop = [x[i] + y[i] for i in range(n)]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
z_vec = x + y
t_vec = time.perf_counter() - t0

print(f"Python loop: {t_loop*1000:8.1f} ms")
print(f"NumPy      : {t_vec*1000:8.1f} ms   (~{t_loop/t_vec:.0f}x faster)")

### Automatic type promotion
Mixed inputs are promoted to a type that can represent all of them.

In [ ]:
i = np.array([1, 2, 3])          # int64
f = np.array([0.5, 0.5, 0.5])    # float64
print((i + f).dtype)             # float64

But promotion is not always lossless: float64 has only 52 mantissa bits, so very large integers get rounded.

In [ ]:
big = np.array([2**53 + 1], dtype=np.int64)
print("int64  :", big[0])
print("float64:", int(big.astype(np.float64)[0]))   # 2**53 + 1 cannot be represented exactly

## dtype

A floating-point number is made of 3 parts: sign, exponent, mantissa (fraction).

In [ ]:
dtypes = [torch.float64, torch.float32, torch.float16, torch.bfloat16,
          torch.int64, torch.int8, torch.bool]
print(f"{'dtype':<16}{'bytes':>6}")
for dt in dtypes:
    print(f"{str(dt):<16}{torch.tensor([], dtype=dt).element_size():>6}")

### fp16 vs bf16

Both use 2 bytes, but split the bits differently:
- **fp16**: 5 exponent bits → small range (max ≈ 65504), but 10 mantissa bits of precision.
- **bf16**: 8 exponent bits → same range as fp32, but only 7 mantissa bits.

In [ ]:
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    fi = torch.finfo(dt)
    print(f"{str(dt):<16} max={fi.max:<12.4g} eps={fi.eps:.3g}")

print()
print("70000 in fp16 :", torch.tensor(70000.0, dtype=torch.float16))   # overflow -> inf
print("70000 in bf16 :", torch.tensor(70000.0, dtype=torch.bfloat16))  # fits, but rounded
print("1.001 in bf16 :", torch.tensor(1.001, dtype=torch.bfloat16))    # precision lost

# Tensor

PyTorch's version of ndarray, with two extra abilities: it can run on a GPU (`device`), and it can track gradients (`requires_grad`).

In [ ]:
t = torch.randn(32, 784)
print("shape        :", t.shape)
print("dtype        :", t.dtype)
print("device       :", t.device)
print("requires_grad:", t.requires_grad)

t_dev = t.to(device)
print("after .to()  :", t_dev.device)

Converting between NumPy and PyTorch:

In [ ]:
arr = np.arange(6, dtype=np.float32).reshape(2, 3)
ten = torch.from_numpy(arr)     # shares memory with arr
print(ten)
print(ten.numpy())              # back to NumPy (CPU tensors only)

### A tensor is a multi-dimensional array

| ndim | name | example in this lecture | shape |
|---|---|---|---|
| 0 | scalar | loss | `[]` |
| 1 | vector | `fc1.bias` | `[256]` |
| 2 | matrix | `fc1.weight` | `[256, 784]` |
| 3 | | one RGB image (C, H, W) | `[3, 28, 28]` |
| 4 | | a batch of RGB images (N, C, H, W) | `[32, 3, 28, 28]` |

In [ ]:
examples = {
    "scalar": torch.tensor(3.14),
    "vector": torch.zeros(256),
    "matrix": torch.zeros(256, 784),
    "one RGB image": torch.zeros(3, 28, 28),
    "a batch of RGB images": torch.zeros(32, 3, 28, 28),
}
for name, t in examples.items():
    print(f"{name:<22} ndim={t.ndim}  shape={tuple(t.shape)}")

## How can a tensor be stored in memory?

No matter how many dimensions a tensor has, its data lives in one **contiguous, one-dimensional block** of memory called **storage**.
**shape**, **stride**, and **offset** decide how to interpret it.

### Stride

In [ ]:
x = torch.arange(12).view(3, 4)
print(x)
print("stride:", x.stride())   # (4, 1): next row = skip 4 elements, next column = skip 1

> ⚠️ **PyTorch strides are measured in elements, NumPy strides in bytes.**
> For the same int64 array, NumPy reports `(32, 8)`: one row is 4 elements × 8 bytes = 32 bytes, one column is 8 bytes.

In [ ]:
a = np.arange(12, dtype=np.int64).reshape(3, 4)
print("NumPy strides (bytes)    :", a.strides)
print("PyTorch stride (elements):", torch.from_numpy(a).stride())

### Offset

In [ ]:
y = x[1:, 1:]
print(y)
print("storage_offset:", y.storage_offset())   # 5
print("stride        :", y.stride())           # (4, 1)

# y is a *view*: it shares the same storage as x, no data was copied
print("same storage? ", y.untyped_storage().data_ptr() == x.untyped_storage().data_ptr())

### Put it together

`position = offset + i₀ × stride[0] + i₁ × stride[1] + ... + iₙ × stride[n]`

Let's verify `y[1, 2]` by hand, and then rebuild `y` from nothing but the storage and the metadata.

In [ ]:
storage = x.flatten()           # x is contiguous, so this is exactly its storage order
pos = y.storage_offset() + 1 * y.stride()[0] + 2 * y.stride()[1]
print("position:", pos, "-> value:", storage[pos].item(), "| y[1, 2] =", y[1, 2].item())

# Rebuild y using only (size, stride, offset)
y_rebuilt = torch.as_strided(x, size=(2, 3), stride=(4, 1), storage_offset=5)
print(y_rebuilt)
print("equal:", torch.equal(y, y_rebuilt))

Because `y` shares storage with `x`, writing to `y` changes `x` too:

In [ ]:
x2 = torch.arange(12).view(3, 4)
y2 = x2[1:, 1:]
y2[0, 0] = 100
print(x2)

### Views that only change metadata

Slicing, transpose, and expand do not move any data; they only change shape / stride / offset.

In [ ]:
xt = x.t()                       # transpose: strides are swapped
print("x.t() stride   :", xt.stride())         # (1, 4)
print("is_contiguous  :", xt.is_contiguous())  # False

v = torch.tensor([1, 2, 3]).expand(4, 3)       # repeat a row 4 times without copying
print(v)
print("expand stride  :", v.stride())          # (0, 1): stride 0 = read the same elements again

`.view()` requires a compatible (usually contiguous) layout. On a transposed tensor it fails:

In [ ]:
try:
    xt.view(12)
except RuntimeError as e:
    print("RuntimeError:", str(e)[:90], "...")

print(xt.contiguous().view(12))    # copy into a new contiguous storage, then view
print(xt.reshape(12))              # reshape copies only when it has to
print("contiguous stride:", xt.contiguous().stride())   # (3, 1)

### A batch of images: `[32, 3, 28, 28]`

PyTorch puts channels first (**NCHW**): dim 0 = which image, dim 1 = which channel (R/G/B), dim 2 = row, dim 3 = column.

In [ ]:
x = torch.randn(32, 3, 28, 28)
print("ndim   :", x.ndim)
print("numel  :", x.numel(), f"→ {x.numel() * x.element_size() / 1e6:.2f} MB in fp32")
print("x[0]   :", tuple(x[0].shape), "  image 0")
print("x[0,1] :", tuple(x[0, 1].shape), "     image 0, green channel")
print("x[:,0] :", tuple(x[:, 0].shape), " red channel of all images")
print("stride :", x.stride(), "  next image = skip 3*28*28 =", 3 * 28 * 28, "elements")

MNIST is grayscale, so it has only **1** channel. That is why the DataLoader below gives `(B, 1, 28, 28)`.

# MLP

## Overview: Dataset - MNIST
28×28 grayscale handwritten digits, 60,000 for training and 10,000 for testing.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),                        # uint8 0–255 -> float32 0–1, shape (1, 28, 28)
    transforms.Normalize((0.1307,), (0.3081,)),   # MNIST mean / std
])

train_set = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
print(len(train_set), "training samples,", len(test_set), "test samples")

In [ ]:
raw = datasets.MNIST(root="./data", train=True, download=False)   # no transform: PIL images
img, label = raw[0]
pixels = np.array(img)
print("label:", label, "| shape:", pixels.shape, "| dtype:", pixels.dtype,
      "| range:", pixels.min(), "-", pixels.max())

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for k, ax in enumerate(axes):
    im, lb = raw[k]
    ax.imshow(im, cmap="gray"); ax.set_title(lb); ax.axis("off")
plt.show()

## Build Structure

### 1. Linear Layer

#### Data processing: follow the shape

In [ ]:
batch_size = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=1000,      shuffle=False)

sample, _ = train_set[0]
print("one sample after ToTensor + Normalize:", tuple(sample.shape), sample.dtype)

images, labels = next(iter(train_loader))
print("one batch from DataLoader            :", tuple(images.shape))   # (B, 1, 28, 28)

flat = nn.Flatten()(images)
print("after Flatten                        :", tuple(flat.shape))     # (B, 784)
print("labels                               :", tuple(labels.shape), labels.dtype)

#### How a neuron does
$z = \mathbf{w} \cdot \mathbf{x} + b$

In [ ]:
x1 = flat[0]                    # one sample, (784,)
w = torch.randn(784)
b = torch.randn(())
z = (w * x1).sum() + b          # weighted sum + bias
print(z, "==", w @ x1 + b)

#### How a group of neurons and a batch are handled

`nn.Linear(784, 256)` stores `weight` with shape `(out, in) = (256, 784)` and computes $Z = XW^\top + \mathbf{b}$.

In [ ]:
fc1 = nn.Linear(784, 256)
print("weight:", tuple(fc1.weight.shape), " bias:", tuple(fc1.bias.shape))

Z = fc1(flat)                                   # (64, 784) -> (64, 256)
Z_manual = flat @ fc1.weight.T + fc1.bias       # bias is broadcast to every row
print("output:", tuple(Z.shape))
print("matches manual X @ W.T + b:", torch.allclose(Z, Z_manual, atol=1e-6))

### 2. Stacking Layers

784 → 256 → 128 → 10. The `out_features` of one layer must equal the `in_features` of the next.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)            # (B, 1, 28, 28) -> (B, 784)
        x = self.relu(self.fc1(x))     # (B, 256)
        x = self.relu(self.fc2(x))     # (B, 128)
        return self.fc3(x)             # (B, 10)  logits, no activation

model = MLP()
print(model)

#### How big is our model?

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:<12} {str(tuple(p.shape)):<12} {p.numel():>8,}")

total = sum(p.numel() for p in model.parameters())
print(f"\nTotal = {total:,} parameters ≈ {total * 4 / 1e6:.2f} MB in fp32")

What happens if the dimensions don't match:

In [ ]:
bad = nn.Sequential(nn.Linear(784, 256), nn.Linear(128, 10))   # 256 != 128
try:
    bad(flat)
except RuntimeError as e:
    print("RuntimeError:", e)

### 3. Activation

#### Why it is needed
Without activation, stacked linear layers collapse into one: $W_3(W_2(W_1x+b_1)+b_2)+b_3 = W'x+b'$. Let's check.

In [ ]:
torch.manual_seed(0)
l1, l2, l3 = nn.Linear(784, 256), nn.Linear(256, 128), nn.Linear(128, 10)
deep_linear = lambda x: l3(l2(l1(x)))        # 3 layers, no activation

with torch.no_grad():
    W_ = l3.weight @ l2.weight @ l1.weight                         # (10, 784)
    b_ = l3.weight @ (l2.weight @ l1.bias + l2.bias) + l3.bias     # (10,)
    single = flat @ W_.T + b_

    print("W' shape:", tuple(W_.shape))
    print("3 linear layers == 1 linear layer:",
          torch.allclose(deep_linear(flat), single, atol=1e-5))

#### Some common activation functions

In [ ]:
s = torch.linspace(-4, 4, 200)
acts = {"ReLU": torch.relu, "Tanh": torch.tanh, "GELU": F.gelu,
        "Sigmoid": torch.sigmoid, "SiLU": F.silu}

plt.figure(figsize=(7, 4))
for name, fn in acts.items():
    plt.plot(s, fn(s), label=name)
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.legend(); plt.title("Activation functions"); plt.show()

print("element-wise, shape unchanged:", tuple(torch.relu(Z).shape) == tuple(Z.shape))

## Run Training

### 4. Forward

In [ ]:
logits = model(images)
print("input :", tuple(images.shape))
print("logits:", tuple(logits.shape))
print(logits[0])

### 5. Loss

#### Why not use accuracy instead?
Target is **3**. Rebuild the table from the notes: accuracy jumps like a staircase, loss decreases smoothly.

In [ ]:
rows = {  # phase: (P(3), P(8), P(others) spread over the other 8 classes)
    "A": (0.30, 0.50, 0.20),
    "B": (0.40, 0.45, 0.15),
    "C": (0.45, 0.40, 0.15),
    "D": (0.90, 0.05, 0.05),
}
target = torch.tensor(3)

print(f"{'Phase':<6}{'P(3)':>6}{'P(8)':>6}{'Pred':>6}{'Acc':>5}{'Loss':>7}")
for phase, (p3, p8, rest) in rows.items():
    probs = torch.full((10,), rest / 8)
    probs[3], probs[8] = p3, p8
    logit = probs.log()                                   # softmax(log p) = p
    loss = F.cross_entropy(logit.unsqueeze(0), target.unsqueeze(0))
    pred = probs.argmax().item()
    print(f"{phase:<6}{p3:>6.2f}{p8:>6.2f}{pred:>6}{int(pred == 3):>5}{loss.item():>7.2f}")

#### From logits to probability: Softmax
$p_k = \dfrac{e^{z_k}}{\sum_j e^{z_j}}$ — note that softmax must know **which dimension** to normalize over.

In [ ]:
probs = torch.softmax(logits, dim=1)
print("each row sums to 1:", probs.sum(dim=1)[:4])
print("all > 0           :", bool((probs > 0).all()))

manual = logits.exp() / logits.exp().sum(dim=1, keepdim=True)
print("matches formula   :", torch.allclose(probs, manual, atol=1e-6))

#### Cross-entropy loss
$\mathcal{L} = -\frac{1}{B}\sum_i \log p^{(i)}_{y_i}$

`nn.CrossEntropyLoss` takes **raw logits** and applies (log-)softmax internally.

In [ ]:
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)

log_p = torch.log_softmax(logits, dim=1)
manual = -log_p[torch.arange(len(labels)), labels].mean()
print("nn.CrossEntropyLoss:", loss.item())
print("manual             :", manual.item())
print("an untrained 10-class model should be near -ln(1/10) =", np.log(10).round(3))

### 6. Backward

#### Reproduce the hand-calculated example
`x ──×w₁──▶ a ──ReLU──▶ h ──×w₂──▶ z ──▶ L = (z − y)²` with $x=2, w_1=0.5, w_2=4, y=1$.

In [ ]:
x_ = torch.tensor(2.0)
y_ = torch.tensor(1.0)
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(4.0, requires_grad=True)

a = w1 * x_;  a.retain_grad()
h = torch.relu(a);  h.retain_grad()
z = w2 * h;  z.retain_grad()
L = (z - y_) ** 2

print("Forward : a =", a.item(), " h =", h.item(), " z =", z.item(), " L =", L.item())
L.backward()
print("Backward: dL/dz  =", z.grad.item())
print("          dL/dw2 =", w2.grad.item())
print("          dL/dh  =", h.grad.item())
print("          dL/da  =", a.grad.item())
print("          dL/dw1 =", w1.grad.item())

#### How backward works in a linear layer
For $Z = H_{in}W^\top + b$, given the upstream gradient $\partial Z$:
- weight gradient $\partial W = (\partial Z)^\top H_{in}$
- gradient passed back $\partial H_{in} = \partial Z\, W$

In [ ]:
H_in = torch.randn(64, 784, requires_grad=True)
layer = nn.Linear(784, 256)
Zl = layer(H_in)

dZ = torch.randn_like(Zl)      # pretend this came from the layers above
Zl.backward(dZ)

print("dW  == dZ.T @ H_in :", torch.allclose(layer.weight.grad, dZ.T @ H_in, atol=1e-4))
print("dH  == dZ @ W      :", torch.allclose(H_in.grad, dZ @ layer.weight, atol=1e-4))
print("db  == dZ.sum(0)   :", torch.allclose(layer.bias.grad, dZ.sum(0), atol=1e-4))

#### ReLU passes the gradient only where the forward input was > 0

In [ ]:
a = torch.tensor([-2.0, -0.5, 0.5, 3.0], requires_grad=True)
torch.relu(a).sum().backward()
print("input:", a.data)
print("grad :", a.grad)

### 7. Update

$\theta \leftarrow \theta - \eta \cdot \dfrac{\partial \mathcal{L}}{\partial \theta}$

Doing it by hand vs. letting an optimizer do it gives the same result:

In [ ]:
torch.manual_seed(0)
m_hand = nn.Linear(4, 1)
m_opt = nn.Linear(4, 1); m_opt.load_state_dict(m_hand.state_dict())
inp, tgt, lr = torch.randn(8, 4), torch.randn(8, 1), 0.1

# by hand
F.mse_loss(m_hand(inp), tgt).backward()
with torch.no_grad():                    # updates should not be tracked by autograd
    for p in m_hand.parameters():
        p -= lr * p.grad

# with an optimizer
opt = torch.optim.SGD(m_opt.parameters(), lr=lr)
F.mse_loss(m_opt(inp), tgt).backward()
opt.step()

print("same result:", all(torch.allclose(p, q) for p, q in zip(m_hand.parameters(), m_opt.parameters())))

> ⚠️ PyTorch **accumulates** gradients in `.grad`. That's why every iteration calls `optimizer.zero_grad()` first.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
for k in range(3):
    (3 * w).backward()
    print(f"after backward #{k+1}: w.grad = {w.grad.item()}")   # 3, 6, 9 ...

### 8. Train loop

forward → loss → backward → update, repeated over every batch, for several epochs.

In [ ]:
print("iterations per epoch:", len(train_loader))   # 60000 / 64 ≈ 938

In [ ]:
torch.manual_seed(0)
model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
num_epochs = 3

def evaluate(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():                     # inference: forward only
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(dim=1) == y).sum().item()
    return correct / len(loader.dataset)

history = []
for epoch in range(num_epochs):
    model.train()
    for it, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)

        logits = model(x)                 # 4. forward
        loss = criterion(logits, y)       # 5. loss
        optimizer.zero_grad()             #    clear old gradients
        loss.backward()                   # 6. backward
        optimizer.step()                  # 7. update

        history.append(loss.item())
        if it % 200 == 0:
            print(f"epoch {epoch+1} | iter {it:4d} | loss {loss.item():.4f}")

    print(f"==> epoch {epoch+1} test accuracy: {evaluate(model, test_loader):.2%}")

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(history, alpha=0.4, label="per iteration")
k = 50
plt.plot(np.convolve(history, np.ones(k) / k, mode="valid"), label=f"moving avg ({k})")
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend(); plt.title("Training loss")
plt.show()

#### Look at some predictions

In [ ]:
model.eval()
x, y = next(iter(test_loader))
with torch.no_grad():
    pred = model(x.to(device)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for k, ax in enumerate(axes.flat):
    ax.imshow(x[k, 0], cmap="gray")
    ax.set_title(f"pred {pred[k].item()} / true {y[k].item()}",
                 color="green" if pred[k] == y[k] else "red", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()

# Lab

### Basic
_TBD_

### Advanced
_TBD_